In [2]:
from pathlib import Path 
import math
import pandas as pd
import pickle
import meds_reader
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import timedelta


disease_dir = Path("/shared/share_mala/zj2398/mimic/regression/cohort/regression_labels_nona")
task="bilirubin"
meds_path = "/user/zj2398/cache/mimic/meds_v0.6_reader"

disease_path = disease_dir/f"{task}_regression.parquet"
# df = pd.read_parquet(disease_path)

database = meds_reader.SubjectDatabase(meds_path)
# print(database[10000032])
# print(disease_file)
# Ensure types
for parquet_file in disease_dir.glob("*.parquet"):
    print(parquet_file.name)
    df = pd.read_parquet(parquet_file)
    # print(f"parquet_file {len(df)}")
    df["patient_id"] = df["patient_id"].astype(int)
    df["time"] = pd.to_datetime(df["time"], errors="coerce")

    # 2) Build unique patient list from the dataframe (so we only fetch what we need)
    all_pids = df["patient_id"].unique().tolist()

    # --- Worker function: open SubjectDatabase inside the process, compute first non-None event time per patient ---
    def first_event_times_for_chunk(pids_chunk, meds_path_str):
        from datetime import datetime
        import meds_reader as _mr

        db = _mr.SubjectDatabase(meds_path_str)

        out = {}
        for pid in pids_chunk:
            try:
                subj = db[pid]
            except KeyError:
                # no subject -> skip
                continue
            first_t = None
            for ev in subj.events:
                # ev.time can be None; keep the earliest non-None
                if ev.time is not None and ev.code != "MEDS_BIRTH":
                    first_t = ev.time
                    break
                    # if (first_t is None) or (ev.time < first_t):
                    #     first_t = ev.time
            if first_t is not None:
                out[pid] = pd.Timestamp(first_t)  # normalize to pandas Timestamp
        return out

    # 3) Parallelize over patient chunks (each worker builds its own SubjectDatabase handle)
    def chunked(iterable, size):
        for i in range(0, len(iterable), size):
            yield iterable[i:i+size]

    pid_chunks = list(chunked(all_pids, 1000))
    pid_to_first = {}

    with ProcessPoolExecutor(max_workers=128) as ex:
        futs = [ex.submit(first_event_times_for_chunk, ch, str(meds_path)) for ch in pid_chunks]
        for fut in as_completed(futs):
            pid_to_first.update(fut.result())

    # 4) Map first event times back to the dataframe
    df["first_event_time"] = df["patient_id"].map(pid_to_first)

    # Drop rows for patients with no valid first event time
    df = df.dropna(subset=["first_event_time"])

    # 5) Keep only rows within the first year after the first event (and not earlier)
    observation_window = pd.Timedelta(days=0)
    mask = df["time"] >= (df["first_event_time"]+observation_window)
    # print(mask,sum(mask))
    # print(df)
    filtered = df.loc[mask, ["patient_id", "time", "numeric_value", "unit"]]  # keep any columns you want
    # print(filtered)
    # print(filtered.groupby("patient_id",group_keys=False))
    filtered = filtered.groupby("patient_id",group_keys=False).sample(n=1,random_state=42)
    filtered = filtered.sort_values(["patient_id", "time"])
    filtered = filtered.rename(columns={
    "patient_id": "subject_id",
    "time": "prediction_time",
    "numeric_value":"numerical_value"
    })
    print(filtered)
    # (Optional) Save
    # # filtered.to_parquet("/path/to/filtered.parquet", index=False)
    store_path = "/shared/share_mala/zj2398/mimic/regression/cohort/regression_labels_random_drop/"+ parquet_file.name
    filtered.to_parquet(store_path)

    print(f"Kept {len(filtered):,} of {len(df):,} rows within {observation_window} of first event.")
    # break
# print(disease_file.iloc[511934]["numeric_value"].dtype())
# print(disease_file["numeric_value"] != float('nan'))
# break

RuntimeError: filesystem error: cannot get file size: No such file or directory [/user/zj2398/cache/mimic/meds_v0.6_reader/subject_id]

In [ ]:
from pathlib import Path
import pandas as pd
import pickle as pl
import numpy as np

    
def find_overlapping_and_unique_rows(df1, df2, columns):
    """
    Find overlapping and unique rows between two DataFrames based on specified columns.
    """
    
    # Find overlapping rows (inner join)
    overlapping = df1.merge(df2[columns], on=columns, how='inner')
    
    # Find unique rows in df1 (left only)
    df1_with_indicator = df1.merge(df2[columns], on=columns, how='left', indicator=True)
    unique_df1 = df1_with_indicator[df1_with_indicator['_merge'] == 'left_only'].drop('_merge', axis=1)
    
    # Find unique rows in df2 (right only)
    df2_with_indicator = df2.merge(df1[columns], on=columns, how='left', indicator=True)
    unique_df2 = df2_with_indicator[df2_with_indicator['_merge'] == 'left_only'].drop('_merge', axis=1)
    
    return overlapping, unique_df1, unique_df2


feature_dir = Path("/shared/share_mala/zj2398/mimic/regression/motor/features")
label_dir = Path("/shared/share_mala/zj2398/mimic/regression/cohort/regression_labels_1_month")
task = "bilirubin"
feature_path = feature_dir/f"{task}_motor.pkl"
with open(feature_path,"rb") as f:
    feature = pl.load(f)
# df = pd.DataFrame(list(feature.items()), columns=['key', 'value'])

# print(feature)
# feature_df = pd.DataFrame({
#         "subject_id": feature['subject_ids'],
#         "prediction_time":feature['feature_times'],
#         "features": list(feature['features'])
#     })
# # df = pd.DataFrame([feature])

for task in ["bilirubin","creatinine","pao2","platelets"]:
    print(task)
    label_path = label_dir/task/f"{task}.parquet"
    label_df = pd.read_parquet(label_path)
    print(label_df)

# print(feature_df)
# print(label_df)

# overlapping,unique1,unique2 = find_overlapping_and_unique_rows(feature_df, label_df, ["subject_id","prediction_time"])
# print(f"overlapping features {overlapping}")
# print(f"unique df1 {unique1}")
# print(f"unique df2 {unique2}")

bilirubin
         subject_id     prediction_time  target_value   unit
1          10000032 2180-05-06 23:16:00           1.6  mg/dL
16         10000084 2160-12-28 05:02:00           0.6  mg/dL
18         10000117 2176-02-08 22:43:00           1.1  mg/dL
25         10000285 2161-11-08 17:44:00           0.7  mg/dL
27         10000560 2189-06-26 14:30:00           0.3  mg/dL
...             ...                 ...           ...    ...
1174777    19999625 2138-10-06 15:21:00           0.7  mg/dL
1174779    19999636 2187-09-15 19:28:00           0.5  mg/dL
1174784    19999784 2119-07-31 09:56:00           0.3  mg/dL
1174891    19999828 2149-01-08 10:47:00           0.4  mg/dL
1174896    19999840 2164-09-12 09:10:00           0.8  mg/dL

[95607 rows x 4 columns]
creatinine
         subject_id     prediction_time  target_value   unit
1          10000032 2180-05-06 23:16:00           0.3  mg/dL
24         10000084 2160-12-27 20:13:00           0.8  mg/dL
27         10000117 2176-02-08 22:43:0

In [4]:
# code_dict = {"LOINC/26515-7//UCUM/10*9/L":"Platelets [#/volume] in Blood","LOINC/2160-0//UCUM/mg/dL":"Creatinine [Mass/volume] in Serum or Plasma"
#              ,"LOINC/1968-7//UCUM/mg/dL":"Bilirubin.direct [Mass/volume] in Serum or Plasma","LOINC/2708-6//UCUM/%":"Oxygen saturation in Arterial blood"}

'''
Goal: Get the distribution of inter-event time for four codes. Process each patient in parallel and collect inter-event time. Plot each distribution
'''
import meds_reader
import functools

code_dict = {"LOINC/26515-7//UCUM/10*9/L":"Platelets","LOINC/2160-0//UCUM/mg/dL":"Creatinine"
             ,"LOINC/1968-7//UCUM/mg/dL":"Bilirubin","LOINC/2708-6//UCUM/%":"Spo2"}

meds_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader/"
LABELS = list(code_dict.values())

def get_interval(subject_iter,code_dict):
    # interval_list = {"Platelets":[],"Creatinine":[],"Bilirubin":[],"Spo2":[]}
    interval_time_list = {lab:[] for lab in list(code_dict.values())}
    for subject in subject_iter:
        # interval_time_list_sub = {lab:[] for lab in list(code_dict.values)}
        last_time = {lab: None for lab in list(code_dict.keys())}
        # print(f"subject is {subject}")
        for event in subject.events:
            # the event belongs to our interested lab items
            if event.code in code_dict and event.time is not None:
                prev_time = last_time[event.code]
                if prev_time is not None:
                    interval_time_list[code_dict[event.code]].append((event.time-prev_time).total_seconds()/86400.0)
                last_time[event.code] = event.time
        print(interval_time_list)
    return interval_time_list

def aggre(init,chunk):
    for k in init:
        init[k].extend(chunk[k])
    return init


meds_database = meds_reader.SubjectDatabase(meds_path,num_threads=1)
init_result = {lab: [] for lab in LABELS}
all_interval_days = functools.reduce(aggre,meds_database.map(functools.partial(get_interval,code_dict=code_dict)),init_result)



KeyboardInterrupt: 

In [ ]:
# code_dict = {"LOINC/26515-7//UCUM/10*9/L":"Platelets [#/volume] in Blood","LOINC/2160-0//UCUM/mg/dL":"Creatinine [Mass/volume] in Serum or Plasma"
#              ,"LOINC/1968-7//UCUM/mg/dL":"Bilirubin.direct [Mass/volume] in Serum or Plasma","LOINC/2708-6//UCUM/%":"Oxygen saturation in Arterial blood"}

'''
Goal: Get the distribution of inter-event time for four codes. Process each patient in parallel and collect inter-event time. Plot each distribution
'''
import meds_reader
import functools
import numpy as np
import matplotlib.pyplot as plt

code_dict = {"LOINC/26515-7//UCUM/10*9/L":"Platelets","LOINC/2160-0//UCUM/mg/dL":"Creatinine"
             ,"LOINC/1968-7//UCUM/mg/dL":"Bilirubin","LOINC/2708-6//UCUM/%":"Spo2"}

LABELS = list(code_dict.values())

def get_interval(subject_iter,code_dict):
    # interval_list = {"Platelets":[],"Creatinine":[],"Bilirubin":[],"Spo2":[]}
    interval_time_list = {lab:[] for lab in list(code_dict.values())}
    for subject in subject_iter:
        # interval_time_list_sub = {lab:[] for lab in list(code_dict.values)}
        last_time = {lab: None for lab in list(code_dict.keys())}
        # print(f"subject is {subject}")
        for event in subject.events:
            # the event belongs to our interested lab items
            if event.code in code_dict and event.time is not None:
                prev_time = last_time[event.code]
                if prev_time is not None:
                    interval_time_list[code_dict[event.code]].append((event.time-prev_time).total_seconds()/86400.0)
                last_time[event.code] = event.time
        # print(interval_time_list)
    return interval_time_list

def aggre(init,chunk):
    for k in init:
        init[k].extend(chunk[k])
    return init


def plot_distributions_days(all_intervals_days, bins=80, clip_q=0.99):
    for lab, vals in all_intervals_days.items():
        x = np.asarray(vals, dtype=float)
        x = x[np.isfinite(x)]
        x = x[x >= 0]

        if x.size == 0:
            print(f"[warn] No intervals for {lab}")
            continue

        xmax = np.quantile(x, clip_q)
        x_plot = x[x <= xmax]

        plt.figure()
        plt.hist(x_plot, bins=bins, density=True)
        plt.title(f"Inter-event time: {lab}")
        plt.xlabel("Days")
        plt.ylabel("Density")
        plt.tight_layout()
        plt.show()



def main():
    meds_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader/"
    db = meds_reader.SubjectDatabase(meds_path, num_threads=32)

    init_result = {lab: [] for lab in LABELS}
    worker = functools.partial(get_interval, code_dict=code_dict)

    all_interval_days = functools.reduce(aggre, db.map(worker), init_result)
    print({k: len(v) for k, v in all_interval_days.items()})
    plot_distributions_days(all_interval_days)

    db.terminate()  # important to clean up processes

if __name__ == "__main__":
    main()

/data/anaconda3/envs/motor_chao/lib/python3.11/site-packages/meds_reader/__init__.py:205: UserWarning: Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.
  warnings.warn("Warning: You are using meds_reader multiprocessing within a Jupyter notebook. This does not reliably work and your code might randomly fail.")
Process SpawnProcess-7:
Traceback (most recent call last):
  File "/data/anaconda3/envs/motor_chao/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/data/anaconda3/envs/motor_chao/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/data/anaconda3/envs/motor_chao/lib/python3.11/site-packages/meds_reader/__init__.py", line 141, in _runner
    map_func = pickle.loads(map_func_str)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'get_interval' on <module '__ma

KeyboardInterrupt: 

In [ ]:
code_dict = {"LOINC/26515-7//UCUM/10*9/L":"Platelets","LOINC/2160-0//UCUM/mg/dL":"Creatinine"
             ,"LOINC/1968-7//UCUM/mg/dL":"Bilirubin","LOINC/2708-6//UCUM/%":"spo2"}
print(code_dict.value())

In [ ]:
import pandas as pd
parquet_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3/visit_occurrence/part-00000-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet"
visit_parquet = pd.read_parquet(parquet_path)
print(len(visit_parquet))
filter = visit_parquet[visit_parquet["visit_concept_id"]==9202]
print(len(filter))

print(type(filter["visit_start_datetime"][0]))



3664160
2652088
         visit_occurrence_id  person_id  visit_concept_id visit_start_date  \
0                    1240047          8              9202       1986-03-12   
1                     999809         28              9202       2011-03-16   
4                    1287950         48              9202       2017-12-07   
5                    2232895         48              9202       2006-02-23   
7                    1217161         68              9202       2019-08-01   
...                      ...        ...               ...              ...   
3664154               672307   15624386              9202       2011-04-06   
3664155              1153040   15624386              9202       2011-06-17   
3664156              1609069   15624386              9202       2011-06-15   
3664157               667621   15624386              9202       2011-07-26   
3664159              1824432   15624400              9202       2004-01-18   

                visit_start_datetime visit_end_

In [1]:
code_dict = {
    "LOINC/26515-7//UCUM/10*9/L": "Platelets",
    "LOINC/2160-0//UCUM/mg/dL": "Creatinine",
    "LOINC/1968-7//UCUM/mg/dL": "Bilirubin",
    "LOINC/2708-6//UCUM/%": "Spo2",
}

import meds_reader
LABELS = list(code_dict.values())
print(LABELS)
meds_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader/"
meds_database = meds_reader.SubjectDatabase(meds_path)
for id in meds_database:
    for event in meds_database[id].events:
        print(event.time, type(event.time))
        # break
    break

['Platelets', 'Creatinine', 'Bilirubin', 'Spo2']
2015-12-27 00:00:00 <class 'datetime.datetime'>
2015-12-27 00:00:00 <class 'datetime.datetime'>
2015-12-27 00:00:00 <class 'datetime.datetime'>
2015-12-27 00:00:00 <class 'datetime.datetime'>
2016-01-12 00:00:00 <class 'datetime.datetime'>
2016-01-12 00:00:00 <class 'datetime.datetime'>
2016-01-12 09:26:50.453000 <class 'datetime.datetime'>
2016-01-12 12:00:00 <class 'datetime.datetime'>
2016-01-12 12:36:30.063000 <class 'datetime.datetime'>
2016-01-12 12:36:30.070000 <class 'datetime.datetime'>
2016-01-12 12:50:56.300000 <class 'datetime.datetime'>
2016-01-12 12:50:56.307000 <class 'datetime.datetime'>
2016-01-12 12:50:56.310000 <class 'datetime.datetime'>
2016-01-12 12:50:56.317000 <class 'datetime.datetime'>
2016-01-12 12:51:20.290000 <class 'datetime.datetime'>
2016-01-12 12:51:20.290000 <class 'datetime.datetime'>


In [9]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

KEEP_COLS = ["person_id", "visit_concept_id", "visit_start_datetime", "visit_end_datetime"]

def combine_visit_parquets(parquet_dir_path: str, out_parquet_path: str, exclude_visit_concept_id: int = 9202):
    parquet_dir = Path(parquet_dir_path)
    files = sorted(parquet_dir.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No .parquet files found under: {parquet_dir}")

    writer = None
    n_in, n_out = 0, 0

    try:
        for fp in files:
            # Read only needed columns
            table = pq.read_table(fp, columns=KEEP_COLS)

            n_in += table.num_rows

            # Filter out visit_concept_id == 9202
            mask = pc.not_equal(table["visit_concept_id"], pa.scalar(exclude_visit_concept_id, pa.int64()))
            table = table.filter(mask)

            # Skip empty shards
            if table.num_rows == 0:
                continue

            # Initialize writer with the first non-empty table's schema
            if writer is None:
                writer = pq.ParquetWriter(out_parquet_path, table.schema, compression="snappy")

            writer.write_table(table)
            n_out += table.num_rows

    finally:
        if writer is not None:
            writer.close()

    print(f"Done. Input rows: {n_in:,}; Output rows (filtered): {n_out:,}")
    print(f"Wrote: {out_parquet_path}")


if __name__ == "__main__":
    parquet_dir_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3/visit_occurrence/"
    out_parquet_path = "/data2/processed_datasets/zj2398/cumc_data/outpatient_visits.parquet"
    combine_visit_parquets(parquet_dir_path, out_parquet_path, exclude_visit_concept_id=9202)


Done. Input rows: 58,616,435; Output rows (filtered): 16,173,772
Wrote: /data2/processed_datasets/zj2398/cumc_data/outpatient_visits.parquet


In [ ]:
"""
Task 1: Combine all parquet files containing visit data, filtering for visit_concept_id=9202

This script efficiently processes multiple parquet files by:
1. Reading only necessary columns to reduce memory footprint
2. Filtering during read to minimize data in memory
3. Using PyArrow for faster I/O operations
4. Concatenating tables in batches to avoid memory spikes
"""

import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc
from pathlib import Path
from typing import Union

def combine_visit_parquets_pyarrow(parquet_dir_path: Union[str, Path], 
                                    output_path: Union[str, Path]) -> pa.Table:
    """
    Combine visit parquet files using PyArrow (faster, more memory efficient).
    
    Args:
        parquet_dir_path: Directory containing input parquet files
        output_path: Path to save the combined output parquet file
    
    Returns:
        Combined PyArrow Table with filtered visits
    
    Efficiency optimizations:
    - PyArrow is 2-5x faster than Pandas for parquet I/O
    - Column pruning happens at read time (minimal memory allocation)
    - Filter pushdown applied immediately (avoids materializing unwanted rows)
    - Zero-copy operations where possible
    """
    parquet_dir = Path(parquet_dir_path)
    parquet_files = sorted(parquet_dir.glob("*.parquet"))
    
    if not parquet_files:
        raise ValueError(f"No parquet files found in {parquet_dir_path}")
    
    print(f"Found {len(parquet_files)} parquet files to process")
    
    # Define columns we need
    columns_to_keep = ['person_id', 'visit_concept_id', 
                       'visit_start_datetime', 'visit_end_datetime']
    
    # Process each file with PyArrow
    filtered_tables = []
    total_rows_before = 0
    total_rows_after = 0
    
    for i, file in enumerate(parquet_files, 1):
        # Read as PyArrow table (faster than Pandas)
        # Column pruning happens at read time
        table = pq.read_table(str(file), columns=columns_to_keep)
        rows_before = table.num_rows
        
        # Apply filter using PyArrow compute (very fast)
        # This creates a boolean mask without materializing filtered data yet
        mask = pc.equal(table['visit_concept_id'], 9202)
        filtered_table = table.filter(mask)
        rows_after = filtered_table.num_rows
        
        total_rows_before += rows_before
        total_rows_after += rows_after
        
        if rows_after > 0:
            filtered_tables.append(filtered_table)
        
        print(f"  File {i}/{len(parquet_files)}: {file.name} - "
              f"kept {rows_after:,} / {rows_before:,} rows")
    
    # Concatenate all tables (zero-copy when possible)
    print(f"\nCombining {len(filtered_tables)} non-empty tables...")
    combined_table = pa.concat_tables(filtered_tables)
    
    print(f"Total rows: {total_rows_after:,} / {total_rows_before:,} "
          f"({100*total_rows_after/total_rows_before:.1f}% kept)")
    print(f"Memory usage: {combined_table.nbytes / 1024**2:.2f} MB")
    
    # Write to parquet with compression
    pq.write_table(combined_table, output_path, compression='snappy')
    print(f"\nSaved combined data to: {output_path}")
    
    return combined_table


# Example usage
if __name__ == "__main__":
    # Configuration
    PARQUET_DIR = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3/visit_occurrence"  # UPDATE THIS
    OUTPUT_FILE = "/data2/processed_datasets/zj2398/cumc_data/outpatient_visits.parquet"  # UPDATE THIS
    
    # Choose method based on your needs:
    
    # Option 1: PyArrow (recommended for large files - faster, more efficient)
    print("=" * 70)
    print("Using PyArrow method (recommended for large datasets)")
    print("=" * 70)
    result_table = combine_visit_parquets_pyarrow(PARQUET_DIR, OUTPUT_FILE)
    
    # Convert to pandas if needed for downstream analysis
    # result_df = result_table.to_pandas()
    
    # Option 2: Pandas (simpler, better for smaller files)
    # print("=" * 70)
    # print("Using Pandas method")
    # print("=" * 70)
    # result_df = combine_visit_parquets_pandas(PARQUET_DIR, OUTPUT_FILE)

Using PyArrow method (recommended for large datasets)
Found 16 parquet files to process
  File 1/16: part-00000-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,088 / 3,664,160 rows
  File 2/16: part-00001-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,142 / 3,663,364 rows
  File 3/16: part-00002-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,654,256 / 3,664,569 rows
  File 4/16: part-00003-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,806 / 3,663,183 rows
  File 5/16: part-00004-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,251 / 3,663,024 rows
  File 6/16: part-00005-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,105 / 3,662,795 rows
  File 7/16: part-00006-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,163 / 3,663,655 rows
  File 8/16: part-00007-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,650,256 / 3,663,6

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import Union, List, Tuple
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
import time


def process_single_file(file_path: str, columns_to_keep: List[str]) -> Tuple[pd.DataFrame, int, int]:
    """
    Worker function to process a single parquet file.
    
    Args:
        file_path: Path to the parquet file
        columns_to_keep: List of column names to read
    
    Returns:
        Tuple of (filtered_df, rows_before, rows_after)
    """
    # Read parquet file
    df = pd.read_parquet(file_path, columns=columns_to_keep)
    rows_before = len(df)
    
    # Parse datetime columns - assuming consistent 7-decimal format
    # Truncate .NNNNNNN to .NNNNNN (7 decimals to 6 decimals)
    for col in ['visit_start_datetime', 'visit_end_datetime']:
        if df[col].dtype == 'object':  # String type
            # Fast string truncation: keep everything before last decimal digit
            df[col] = df[col].str[:-1]  # Remove last character (7th decimal)
            df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S.%f')
        else:
            df[col] = pd.to_datetime(df[col])
            if df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_localize(None)
    
    # Ensure integer types
    df['person_id'] = df['person_id'].astype('int64')
    df['visit_concept_id'] = df['visit_concept_id'].astype('int64')
    
    # Filter for outpatient visits
    df_filtered = df[df['visit_concept_id'] == 9202].copy()
    rows_after = len(df_filtered)
    
    return df_filtered, rows_before, rows_after


def combine_visit_parquets_parallel(parquet_dir_path: Union[str, Path], 
                                     output_path: Union[str, Path],
                                     max_workers: int = None) -> pd.DataFrame:
    """
    Combine visit parquet files in parallel.
    
    Args:
        parquet_dir_path: Directory containing input parquet files
        output_path: Path to save the combined output parquet file
        max_workers: Number of parallel workers (default: cpu_count - 1)
    
    Returns:
        Combined DataFrame with filtered visits
    """
    parquet_dir = Path(parquet_dir_path)
    parquet_files = sorted(parquet_dir.glob("*.parquet"))
    
    if not parquet_files:
        raise ValueError(f"No parquet files found in {parquet_dir_path}")
    
    # Default to using all CPUs minus 1
    if max_workers is None:
        max_workers = max(1, cpu_count() - 1)
    
    print(f"Found {len(parquet_files)} parquet files to process")
    print(f"Using {max_workers} parallel workers")
    
    columns_to_keep = ['person_id', 'visit_concept_id', 
                       'visit_start_datetime', 'visit_end_datetime']
    
    start_time = time.time()
    
    # Process files in parallel
    filtered_dfs = []
    total_rows_before = 0
    total_rows_after = 0
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all files for processing
        future_to_file = {
            executor.submit(process_single_file, str(file), columns_to_keep): file 
            for file in parquet_files
        }
        
        # Collect results as they complete
        for i, future in enumerate(as_completed(future_to_file), 1):
            file = future_to_file[future]
            try:
                df_filtered, rows_before, rows_after = future.result()
                
                total_rows_before += rows_before
                total_rows_after += rows_after
                
                if len(df_filtered) > 0:
                    filtered_dfs.append(df_filtered)
                
                print(f"  [{i}/{len(parquet_files)}] {file.name} - "
                      f"kept {rows_after:,} / {rows_before:,} rows")
                
            except Exception as e:
                print(f"  ERROR processing {file.name}: {e}")
                raise
    
    elapsed = time.time() - start_time
    print(f"\nProcessing completed in {elapsed:.2f} seconds")
    print(f"  Average: {elapsed/len(parquet_files):.3f} sec/file")
    
    # Combine all filtered DataFrames
    print(f"\nCombining {len(filtered_dfs)} non-empty DataFrames...")
    combined_df = pd.concat(filtered_dfs, ignore_index=True)
    
    print(f"Total rows: {total_rows_after:,} / {total_rows_before:,} "
          f"({100*total_rows_after/total_rows_before:.1f}% kept)")
    print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Verify dtypes
    print("\nFinal schema:")
    print(combined_df.dtypes)
    
    # Save to parquet
    print(f"\nSaving to {output_path}...")
    combined_df.to_parquet(
        output_path, 
        index=False, 
        engine='pyarrow',
        compression='snappy'
    )
    print(f"Saved combined data successfully!")
    
    return combined_df


# Example usage
if __name__ == "__main__":
    PARQUET_DIR = "xxx"  # UPDATE THIS
    OUTPUT_FILE = "xxx"  # UPDATE THIS
    
    PARQUET_DIR = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3/visit_occurrence"  # UPDATE THIS
    OUTPUT_FILE = "/data2/processed_datasets/zj2398/cumc_data/outpatient_visits_new.parquet"  # UPDATE THIS
    # Run with automatic CPU detection
    result = combine_visit_parquets_parallel(PARQUET_DIR, OUTPUT_FILE,16)
    
    # Or specify number of workers explicitly
    # result = combine_visit_parquets_parallel(PARQUET_DIR, OUTPUT_FILE, max_workers=16)

Found 16 parquet files to process
Using 16 parallel workers
  [1/16] part-00011-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,141 / 3,664,053 rows
  [2/16] part-00009-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,686 / 3,663,948 rows
  [3/16] part-00000-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,088 / 3,664,160 rows
  [4/16] part-00015-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,286 / 3,663,293 rows
  [5/16] part-00012-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,205 / 3,662,681 rows
  [6/16] part-00004-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,251 / 3,663,024 rows
  [7/16] part-00001-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,653,142 / 3,663,364 rows
  [8/16] part-00005-b5bf5dd2-d054-4429-8905-b79c29aa68a8-c000.snappy.parquet - kept 2,652,105 / 3,662,795 rows
  [9/16] part-00002-b5bf5dd2-d054-4429-8905-b79c29aa

In [9]:
import pandas as pd
for task in ["platelets","creatinine","bilirubin","pao2"]:
    regression_path = f"/data2/processed_datasets/mimic/regression/icu_stay_4h/{task}/{task}.parquet"
    regression_file = pd.read_parquet(regression_path)
    # print(regression_file)
    print(len(regression_file))
    print(len(regression_file[regression_file["numerical_value"] != 0]))
    print(len(regression_file["numerical_value"].unique()))
    # break

# regression_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/regression/bilirubin/bilirubin.parquet"
# regression_file = pd.read_parquet(regression_path)
# print(regression_file)

464608
464608
1222
521338
521273
240
138379
138346
593
439054
439052
591


In [7]:
import pandas as pd
# for task in ["bilirubin"]:
for task in ["platelets","creatinine","bilirubin"]:
    regression_path = f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/regression/{task}/{task}.parquet"
    regression_file = pd.read_parquet(regression_path)

    print(len(regression_file))
    print(len(regression_file["numerical_value"].unique()))
    # new_regression_file = regression_file[regression_file["numerical_value"] != 0]
    # # new_regression_file = new_regression_file.rename(columns={"regression_value":"numerical_value"})
    # print(len(new_regression_file))
    # print(new_regression_file)
    # new_regression_file.to_parquet(regression_path)
    # print(len(regression_file[regression_file["numerical_value"] != 0]))

400154
868
419583
1198
249768
126


In [ ]:
import pandas as pd
for task in ["Platelets","Creatinine","Bilirubin"]:
    lab_path = f"/data2/processed_datasets/zj2398/cumc_data/regression/cohort_{task}.parquet"
    lab_visit = pd.read_parquet(lab_path)
    print(len(lab_visit))
    print(lab_visit[["subject_id","event_time"]].drop_duplicates().shape[0])
    # print(lab_visit[["subject_id","event_time"]].drop_duplicates().shape[0])
    print(lab_visit[["subject_id","prediction_time"]].drop_duplicates().shape[0])

266367
40088
266367


In [ ]:
print(lab_visit[["subject_id","event_time"]].drop_duplicates().shape[0])
print(lab_visit[["subject_id","prediction_time"]].drop_duplicates().shape[0])

# "Platelets"
# 400154
# 64037

# "Creatinine"
# 419585
# 78507

# "Bilirubin"
# 266367
# 40088

64037
400154
